# T-Rex JAX/MJX Training (Colab)

Train a T-Rex using **MuJoCo MJX** (JAX-accelerated physics) with a from-scratch PPO
implementation in pure JAX. MJX vectorises thousands of parallel simulations on a single
GPU, giving 10-100x speedups over CPU-based Gymnasium training.

**Requirements:** Colab GPU runtime (A100 recommended).

**Training Stages:**
1. **Balance** – Stand without falling
2. **Locomotion** – Walk and run forward
3. **Bite** – Sprint toward prey and bite

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco mujoco-mjx jax[cuda12] flax optax

import os
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError("GPU not found. Use a GPU Colab runtime.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# Clone mesozoic-labs and install
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
!pip install -e /content/mesozoic-labs -q

from IPython.display import clear_output

clear_output()
print("mesozoic-labs installed.")

In [ ]:
import time

import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import optax

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
USE_GOOGLE_DRIVE = False  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# ============================================================
# Storage Configuration
# ============================================================
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/mesozoic-labs/jax_training")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Outputs will be saved to: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = Path(".")
    print(f"Outputs will be saved to local Colab storage: {OUTPUT_DIR}")

print(f"Google Drive storage: {USE_GOOGLE_DRIVE}")
print(f"Verbose: {VERBOSE}")

## 1. Load T-Rex Model into MJX

In [ ]:
# Load the T-Rex MJCF model
xml_path = "/content/mesozoic-labs/environments/trex/assets/trex.xml"
mj_model = mujoco.MjModel.from_xml_path(xml_path)
mj_data = mujoco.MjData(mj_model)

print("T-Rex Model loaded:")
print(f"  Bodies: {mj_model.nbody}")
print(f"  Joints: {mj_model.njnt}")
print(f"  Actuators (nu): {mj_model.nu}")
print(f"  qpos dim: {mj_model.nq}")
print(f"  qvel dim: {mj_model.nv}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg")
print(f"  Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

# Put model on device (GPU)
mjx_model = mjx.put_model(mj_model)
print(f"\nModel placed on {jax.devices()[0]}")

## 2. Environment Functions

Pure-JAX functions for observation, reward, reset, and step. All are
`jax.vmap`-friendly so they run across thousands of parallel envs.

In [ ]:
# ---------- Constants ----------
FRAME_SKIP = 5
HEALTHY_Z_MIN = 0.4
HEALTHY_Z_MAX = 1.6
MAX_EPISODE_STEPS = 1000

# Body / geom / site IDs (looked up once from the MuJoCo model)
PELVIS_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "pelvis")
FLOOR_GEOM_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
TORSO_GEOM_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "torso")

# Sensor layout (matches MJCF): gyro(3), accel(3), quat(4), r_foot(1), l_foot(1)
S_GYRO = slice(0, 3)
S_ACCEL = slice(3, 6)
S_QUAT = slice(6, 10)
S_RFOOT = 10
S_LFOOT = 11

# Action scaling ranges
CTRL_MIN = jnp.array(mj_model.actuator_ctrlrange[:, 0])
CTRL_MAX = jnp.array(mj_model.actuator_ctrlrange[:, 1])

print(f"Pelvis body id: {PELVIS_ID}")
print(f"Action dim: {mj_model.nu}")

In [ ]:
def get_obs(data):
    """Extract observation vector from MJX data (single env)."""
    qpos = data.qpos[7:]  # exclude root freejoint
    qvel = data.qvel[6:]  # exclude root freejoint
    pelvis_quat = data.sensordata[S_QUAT]
    pelvis_gyro = data.sensordata[S_GYRO]
    pelvis_linvel = data.qvel[:3]
    pelvis_accel = data.sensordata[S_ACCEL]
    foot_contact = jnp.array(
        [
            data.sensordata[S_RFOOT],
            data.sensordata[S_LFOOT],
        ]
    )
    return jnp.concatenate(
        [
            qpos,
            qvel,
            pelvis_quat,
            pelvis_gyro,
            pelvis_linvel,
            pelvis_accel,
            foot_contact,
        ]
    )


def compute_reward(data, action, reward_cfg):
    """Compute scalar reward from MJX data (single env)."""
    # Forward velocity
    forward_vel = data.qvel[0]
    r_forward = reward_cfg["forward_vel_weight"] * forward_vel

    # Alive bonus
    r_alive = reward_cfg["alive_bonus"]

    # Energy penalty
    r_energy = -reward_cfg["energy_penalty_weight"] * jnp.sum(action**2)

    return r_forward + r_alive + r_energy


def is_terminated(data):
    """Check if the T-Rex has fallen."""
    pelvis_z = data.xpos[PELVIS_ID, 2]
    fallen = (pelvis_z < HEALTHY_Z_MIN) | (pelvis_z > HEALTHY_Z_MAX)
    return fallen


def scale_action(action):
    """Scale action from [-1, 1] to actuator control range."""
    return CTRL_MIN + (action + 1.0) * 0.5 * (CTRL_MAX - CTRL_MIN)


# Verify obs dimension
mujoco.mj_forward(mj_model, mj_data)
_test_data = mjx.put_data(mj_model, mj_data)
_test_obs = get_obs(_test_data)
OBS_DIM = _test_obs.shape[0]
ACT_DIM = mj_model.nu
print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
def mjx_step_single(model, data, action):
    """Step one environment: apply action, advance physics, return new data."""
    ctrl = scale_action(action)
    data = data.replace(ctrl=ctrl)

    # Frame skip: step physics multiple times per action
    def body_fn(_, d):
        return mjx.step(model, d)

    data = jax.lax.fori_loop(0, FRAME_SKIP, body_fn, data)
    return data


# Vectorize: model is shared (None), data and action are batched (0)
@jax.jit
def batched_step(model, data_batch, action_batch):
    return jax.vmap(mjx_step_single, in_axes=(None, 0, 0))(model, data_batch, action_batch)


print("Batched step function compiled.")

## 4. Policy Network (Flax)

In [ ]:
class ActorCritic(nn.Module):
    """Shared-backbone actor-critic for PPO."""

    action_dim: int
    hidden_dim: int = 256

    @nn.compact
    def __call__(self, x):
        # Shared backbone
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.relu(x)
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.relu(x)

        # Actor head: mean + log_std
        mean = nn.Dense(self.action_dim)(x)
        log_std = self.param("log_std", nn.initializers.zeros_init(), (self.action_dim,))

        # Critic head
        value = nn.Dense(1)(x)
        value = jnp.squeeze(value, axis=-1)

        return mean, log_std, value


# Initialize
network = ActorCritic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(42)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO Implementation

In [ ]:
def sample_action(params, obs, rng):
    """Sample action from Gaussian policy."""
    mean, log_std, value = network.apply(params, obs)
    std = jnp.exp(log_std)
    noise = jax.random.normal(rng, shape=mean.shape)
    action = mean + std * noise
    action = jnp.clip(action, -1.0, 1.0)

    # Log probability
    log_prob = -0.5 * jnp.sum(
        ((action - mean) / (std + 1e-8)) ** 2 + 2 * log_std + jnp.log(2 * jnp.pi),
        axis=-1,
    )
    return action, log_prob, value


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation."""
    T = rewards.shape[0]
    advantages = jnp.zeros_like(rewards)
    gae = jnp.zeros(rewards.shape[1:])

    def body_fn(i, carry):
        advantages, gae = carry
        t = T - 1 - i
        next_val = jnp.where(t < T - 1, values[t + 1], jnp.zeros_like(gae))
        delta = rewards[t] + gamma * next_val * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages = advantages.at[t].set(gae)
        return advantages, gae

    advantages, _ = jax.lax.fori_loop(0, T, body_fn, (advantages, gae))
    returns = advantages + values
    return advantages, returns


def ppo_loss(params, obs, actions, old_log_probs, advantages, returns, clip_range=0.2, vf_coef=0.5, ent_coef=0.01):
    """PPO clipped surrogate loss. Returns (total_loss, aux_metrics)."""
    mean, log_std, values = jax.vmap(network.apply, in_axes=(None, 0))(params, obs)
    std = jnp.exp(log_std)

    # Log probability of taken actions
    log_probs = -0.5 * jnp.sum(
        ((actions - mean) / (std + 1e-8)) ** 2 + 2 * log_std + jnp.log(2 * jnp.pi),
        axis=-1,
    )

    # Policy loss (clipped)
    ratio = jnp.exp(log_probs - old_log_probs)
    adv_normalized = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    surr1 = ratio * adv_normalized
    surr2 = jnp.clip(ratio, 1 - clip_range, 1 + clip_range) * adv_normalized
    policy_loss = -jnp.mean(jnp.minimum(surr1, surr2))

    # Value loss
    value_loss = jnp.mean((values - returns) ** 2)

    # Entropy bonus
    entropy = 0.5 * jnp.sum(jnp.log(2 * jnp.pi * jnp.e * std**2), axis=-1)
    entropy_bonus = jnp.mean(entropy)

    total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_bonus

    # Auxiliary metrics for logging (not used in gradient computation)
    aux = {
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy_bonus,
        "approx_kl": jnp.mean((ratio - 1) - jnp.log(ratio)),
        "clip_fraction": jnp.mean((jnp.abs(ratio - 1.0) > clip_range).astype(jnp.float32)),
        "mean_std": jnp.mean(std),
    }

    return total_loss, aux


print("PPO functions defined (with decomposed loss metrics).")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters ----------
NUM_ENVS = 2048  # parallel environments
ROLLOUT_LEN = 64  # steps per rollout
NUM_UPDATES = 500  # total PPO updates (increase for full training)
PPO_EPOCHS = 4  # epochs per update
MINIBATCH_SIZE = 512
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.01
FALL_PENALTY = -10.0

# Reward config (Stage 1: balance)
reward_cfg = {
    "forward_vel_weight": 0.0,
    "alive_bonus": 1.0,
    "energy_penalty_weight": 0.0005,
}

print("Training config:")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")

In [ ]:
# Initialize batched environments
rng = jax.random.PRNGKey(42)

# Reset: create initial MJX data for all envs
mujoco.mj_resetData(mj_model, mj_data)
mujoco.mj_forward(mj_model, mj_data)
base_data = mjx.put_data(mj_model, mj_data)


# Replicate across batch with small perturbations
def init_env(rng):
    noise = jax.random.uniform(rng, (mj_model.nq - 7,), minval=-0.01, maxval=0.01)
    qpos = base_data.qpos.at[7:].add(noise)
    return base_data.replace(qpos=qpos)


rngs = jax.random.split(rng, NUM_ENVS)
env_batch = jax.vmap(init_env)(rngs)

# Forward pass to update derived quantities
env_batch = jax.jit(jax.vmap(mjx.forward, in_axes=(None, 0)))(mjx_model, env_batch)

print(f"Initialized {NUM_ENVS} parallel environments.")
print(f"qpos batch shape: {env_batch.qpos.shape}")

In [ ]:
# Optimizer
optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(params)


# JIT-compiled PPO update step (with gradient norm and loss decomposition)
@jax.jit
def ppo_update(params, opt_state, obs, actions, log_probs, advantages, returns):
    (loss, aux), grads = jax.value_and_grad(ppo_loss, has_aux=True)(
        params,
        obs,
        actions,
        log_probs,
        advantages,
        returns,
        clip_range=CLIP_RANGE,
        ent_coef=ENT_COEF,
    )
    grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree.leaves(grads)))
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux, grad_norm


# JIT-compiled batched action sampling
@jax.jit
def batched_sample(params, obs_batch, rng):
    rngs = jax.random.split(rng, obs_batch.shape[0])
    return jax.vmap(sample_action, in_axes=(None, 0, 0))(params, obs_batch, rngs)


# JIT-compiled batched observation + reward
@jax.jit
def batched_obs(data_batch):
    return jax.vmap(get_obs)(data_batch)


@jax.jit
def batched_reward(data_batch, action_batch):
    return jax.vmap(compute_reward, in_axes=(0, 0, None))(data_batch, action_batch, reward_cfg)


@jax.jit
def batched_terminated(data_batch):
    return jax.vmap(is_terminated)(data_batch)


# Reset helper: reset fallen envs to initial state
@jax.jit
def reset_fallen(env_batch, dones, rng):
    """Reset environments where done=True."""
    rngs = jax.random.split(rng, NUM_ENVS)
    fresh = jax.vmap(init_env)(rngs)
    fresh = jax.vmap(mjx.forward, in_axes=(None, 0))(mjx_model, fresh)

    # Select fresh state for done envs, keep existing for others
    def select(fresh_field, existing_field):
        expand = dones.reshape((-1,) + (1,) * (existing_field.ndim - 1))
        return jnp.where(expand, fresh_field, existing_field)

    return jax.tree.map(select, fresh, env_batch)


print("JIT functions ready (with gradient norm tracking).")

In [ ]:
# ---------- Main Training Loop ----------
# Enhanced with: gradient norm tracking, decomposed loss logging,
# periodic checkpointing, best-model tracking, and CSV log for post-hoc analysis.

import csv
import json
import pickle

CHECKPOINT_FREQ = 50  # Save checkpoint every N updates

# Console log frequency based on VERBOSE:
# 0 = only final summary, 1 = every 20 updates (default), 2 = every update
_LOG_INTERVAL = {0: None, 1: 20, 2: 1}.get(VERBOSE, 20)

reward_history = []
loss_history = []
diagnostics_history = []  # Per-update detailed metrics

# Best model tracking
best_reward = -float("inf")
best_params = None
best_update = -1

# CSV log file for easy import into pandas/spreadsheets
csv_path = OUTPUT_DIR / "trex_jax_training_log.csv"
csv_fields = [
    "update", "reward", "total_loss", "policy_loss", "value_loss",
    "entropy", "approx_kl", "clip_fraction", "grad_norm", "mean_std",
    "steps", "sps", "fall_rate",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

print(f"Starting training: {NUM_UPDATES} updates x {ROLLOUT_LEN} steps x {NUM_ENVS} envs")
print(f"Checkpoint frequency: every {CHECKPOINT_FREQ} updates")
print(f"CSV log: {csv_path}")
print("=" * 70)

t_start = time.time()

for update in range(NUM_UPDATES):
    # ---------- Collect rollout ----------
    all_obs, all_actions, all_log_probs, all_values = [], [], [], []
    all_rewards, all_dones = [], []

    for t in range(ROLLOUT_LEN):
        rng, rng_act, rng_reset = jax.random.split(rng, 3)

        obs = batched_obs(env_batch)
        actions, log_probs, values = batched_sample(params, obs, rng_act)

        # Step environments
        env_batch = batched_step(mjx_model, env_batch, actions)

        rewards = batched_reward(env_batch, actions)
        dones = batched_terminated(env_batch)

        # Add fall penalty
        rewards = rewards + dones.astype(jnp.float32) * FALL_PENALTY

        all_obs.append(obs)
        all_actions.append(actions)
        all_log_probs.append(log_probs)
        all_values.append(values)
        all_rewards.append(rewards)
        all_dones.append(dones.astype(jnp.float32))

        # Reset fallen envs
        env_batch = reset_fallen(env_batch, dones, rng_reset)

    # Stack rollout data: (T, NUM_ENVS, ...)
    obs_t = jnp.stack(all_obs)  # (T, N, obs_dim)
    act_t = jnp.stack(all_actions)  # (T, N, act_dim)
    lp_t = jnp.stack(all_log_probs)  # (T, N)
    val_t = jnp.stack(all_values)  # (T, N)
    rew_t = jnp.stack(all_rewards)  # (T, N)
    done_t = jnp.stack(all_dones)  # (T, N)

    # ---------- Compute advantages ----------
    advantages, returns = compute_gae(rew_t, val_t, done_t, GAMMA, GAE_LAMBDA)

    # Flatten: (T * N, ...)
    flat_obs = obs_t.reshape(-1, OBS_DIM)
    flat_act = act_t.reshape(-1, ACT_DIM)
    flat_lp = lp_t.reshape(-1)
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    # ---------- PPO update epochs ----------
    total_samples = flat_obs.shape[0]
    epoch_losses = []
    epoch_aux = []
    epoch_grad_norms = []

    for epoch in range(PPO_EPOCHS):
        rng, rng_perm = jax.random.split(rng)
        perm = jax.random.permutation(rng_perm, total_samples)

        for start in range(0, total_samples, MINIBATCH_SIZE):
            idx = perm[start : start + MINIBATCH_SIZE]
            mb_obs = flat_obs[idx]
            mb_act = flat_act[idx]
            mb_lp = flat_lp[idx]
            mb_adv = flat_adv[idx]
            mb_ret = flat_ret[idx]

            params, opt_state, loss, aux, grad_norm = ppo_update(
                params,
                opt_state,
                mb_obs,
                mb_act,
                mb_lp,
                mb_adv,
                mb_ret,
            )
            epoch_losses.append(float(loss))
            epoch_aux.append({k: float(v) for k, v in aux.items()})
            epoch_grad_norms.append(float(grad_norm))

    avg_reward = float(rew_t.mean())
    avg_loss = np.mean(epoch_losses)
    avg_grad_norm = np.mean(epoch_grad_norms)
    avg_aux = {k: np.mean([a[k] for a in epoch_aux]) for k in epoch_aux[0]}
    fall_rate = float(done_t.sum()) / (ROLLOUT_LEN * NUM_ENVS)

    reward_history.append(avg_reward)
    loss_history.append(avg_loss)
    diagnostics_history.append({
        "reward": avg_reward,
        "loss": avg_loss,
        "grad_norm": avg_grad_norm,
        "fall_rate": fall_rate,
        **avg_aux,
    })

    # Track best model
    if avg_reward > best_reward:
        best_reward = avg_reward
        best_params = jax.device_get(params)
        best_update = update

    # Write to CSV log
    elapsed = time.time() - t_start
    steps_done = (update + 1) * ROLLOUT_LEN * NUM_ENVS
    sps = steps_done / elapsed
    csv_writer.writerow({
        "update": update,
        "reward": f"{avg_reward:.4f}",
        "total_loss": f"{avg_loss:.4f}",
        "policy_loss": f"{avg_aux['policy_loss']:.4f}",
        "value_loss": f"{avg_aux['value_loss']:.4f}",
        "entropy": f"{avg_aux['entropy']:.4f}",
        "approx_kl": f"{avg_aux['approx_kl']:.6f}",
        "clip_fraction": f"{avg_aux['clip_fraction']:.4f}",
        "grad_norm": f"{avg_grad_norm:.4f}",
        "mean_std": f"{avg_aux['mean_std']:.4f}",
        "steps": steps_done,
        "sps": f"{sps:.0f}",
        "fall_rate": f"{fall_rate:.4f}",
    })
    csv_file.flush()

    # Console logging (respects VERBOSE setting)
    if _LOG_INTERVAL is not None and (update % _LOG_INTERVAL == 0 or update == NUM_UPDATES - 1):
        print(
            f"Update {update:4d}/{NUM_UPDATES}  "
            f"reward={avg_reward:+.3f}  loss={avg_loss:.4f}  "
            f"pi_loss={avg_aux['policy_loss']:.4f}  v_loss={avg_aux['value_loss']:.4f}  "
            f"entropy={avg_aux['entropy']:.3f}  kl={avg_aux['approx_kl']:.5f}  "
            f"grad={avg_grad_norm:.3f}  falls={fall_rate:.2%}  "
            f"SPS={sps:,.0f}"
        )

    # Periodic checkpointing
    if (update + 1) % CHECKPOINT_FREQ == 0:
        ckpt_path = OUTPUT_DIR / f"trex_jax_checkpoint_{update + 1}.pkl"
        with open(ckpt_path, "wb") as f:
            pickle.dump({
                "params": jax.device_get(params),
                "update": update + 1,
                "reward_history": reward_history,
                "loss_history": loss_history,
            }, f)
        if _LOG_INTERVAL is not None:
            print(f"  >>> Checkpoint saved: {ckpt_path}")

csv_file.close()

elapsed = time.time() - t_start
total_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print("=" * 70)
print(f"Done! {total_steps:,} steps in {elapsed:.1f}s ({total_steps / elapsed:,.0f} SPS)")
print(f"Best reward: {best_reward:+.4f} at update {best_update}")

# Save final trained parameters and full training history
params_path = OUTPUT_DIR / "trex_jax_params.pkl"
with open(params_path, "wb") as f:
    pickle.dump({
        "params": jax.device_get(params),
        "best_params": best_params,
        "best_reward": best_reward,
        "best_update": best_update,
        "reward_history": reward_history,
        "loss_history": loss_history,
        "diagnostics_history": diagnostics_history,
    }, f)
print(f"Parameters and training history saved to: {params_path}")
print(f"Training log CSV saved to: {csv_path}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Reward
ax = axes[0, 0]
ax.plot(reward_history, "b-", alpha=0.3, label="per-update")
window = min(20, len(reward_history) // 4 + 1)
if window > 1:
    smoothed = np.convolve(reward_history, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(reward_history)), smoothed, "b-", linewidth=2, label=f"{window}-update avg")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Mean Reward")
ax.set_title("Reward")
ax.legend()
ax.grid(True, alpha=0.3)

# Total loss
ax = axes[0, 1]
ax.plot(loss_history, "r-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Total Loss")
ax.grid(True, alpha=0.3)

# Gradient norm
ax = axes[0, 2]
grad_norms = [d["grad_norm"] for d in diagnostics_history]
ax.plot(grad_norms, "g-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Gradient Norm")
ax.set_title("Gradient Norm")
ax.grid(True, alpha=0.3)

# Policy loss vs Value loss
ax = axes[1, 0]
pi_losses = [d["policy_loss"] for d in diagnostics_history]
v_losses = [d["value_loss"] for d in diagnostics_history]
ax.plot(pi_losses, "b-", alpha=0.5, label="Policy loss")
ax.plot(v_losses, "r-", alpha=0.5, label="Value loss")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Policy vs Value Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Entropy and KL divergence
ax = axes[1, 1]
entropies = [d["entropy"] for d in diagnostics_history]
ax.plot(entropies, "purple", alpha=0.7, label="Entropy")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")
ax2 = ax.twinx()
kls = [d["approx_kl"] for d in diagnostics_history]
ax2.plot(kls, "orange", alpha=0.7, label="Approx KL")
ax2.set_ylabel("Approx KL")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# Fall rate
ax = axes[1, 2]
fall_rates = [d["fall_rate"] for d in diagnostics_history]
ax.plot(fall_rates, "brown", alpha=0.5)
if window > 1:
    smoothed_falls = np.convolve(fall_rates, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(fall_rates)), smoothed_falls, "brown", linewidth=2)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Fall Rate")
ax.set_title("Episode Termination Rate")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.suptitle("T-Rex JAX/MJX Training Diagnostics", fontsize=14, fontweight="bold")
plt.tight_layout()
curve_path = OUTPUT_DIR / "trex_jax_training_curves.png"
plt.savefig(curve_path, dpi=150)
print(f"Training curves saved to: {curve_path}")
plt.show()

## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Install with: pip install mediapy")

if _HAS_MEDIAPY:
    # Use the best model (highest reward during training) for video recording
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    # Set up CPU-based MuJoCo renderer
    renderer = mujoco.Renderer(mj_model, height=480, width=640)

    # Reset environment
    mujoco.mj_resetData(mj_model, mj_data)
    mujoco.mj_forward(mj_model, mj_data)

    frames = []
    episode_reward = 0.0

    for step in range(MAX_EPISODE_STEPS):
        # Get observation from CPU data
        cpu_data = mjx.put_data(mj_model, mj_data)
        obs = get_obs(cpu_data)

        # Get deterministic action (use mean, no noise)
        mean, log_std, value = network.apply(video_params, obs)
        action = jnp.clip(mean, -1.0, 1.0)

        # Apply action to CPU sim
        ctrl = np.array(scale_action(action))
        mj_data.ctrl[:] = ctrl
        for _ in range(FRAME_SKIP):
            mujoco.mj_step(mj_model, mj_data)

        # Render frame
        renderer.update_scene(mj_data)
        frames.append(renderer.render())

        # Compute reward
        cpu_data = mjx.put_data(mj_model, mj_data)
        r = float(compute_reward(cpu_data, action, reward_cfg))
        episode_reward += r

        # Check termination
        pelvis_z = mj_data.xpos[PELVIS_ID, 2]
        if pelvis_z < HEALTHY_Z_MIN or pelvis_z > HEALTHY_Z_MAX:
            break

    renderer.close()

    video_path = str(OUTPUT_DIR / "trex_jax_mjx_training.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)

## 9. Next Steps

To continue with curriculum training, update `reward_cfg` for the next stage:

```python
# Stage 2: locomotion
reward_cfg = {
    'forward_vel_weight': 1.0,
    'alive_bonus': 0.5,
    'energy_penalty_weight': 0.001,
}

# Stage 3: bite (add distance-based shaping)
reward_cfg = {
    'forward_vel_weight': 1.0,
    'alive_bonus': 0.1,
    'energy_penalty_weight': 0.001,
}
```

Then re-run the training loop cells. The policy parameters carry over automatically.
After each stage, re-run the video recording cell to capture the new behavior.